In [2]:
import pandas as pd
import firebase_admin
from firebase_admin import credentials, firestore
from google.cloud.firestore_v1 import FieldFilter

In [3]:
# Initialize Firebase
cred = credentials.Certificate('C:\\Users\\mdj20\\Downloads\\watermelon-cup-production-firebase-adminsdk-2rmym-f1467bd3a9.json')
firebase_admin.initialize_app(cred)
db = firestore.client()

In [13]:
# Step 1: Load and clean the CSV into a dict
csv_url = 'https://docs.google.com/spreadsheets/d/1K7DAr85bFv1Q8cEyWYXX2ohIgN6u1u4zETqjuTLumUQ/export?format=csv'
df = pd.read_csv(csv_url, header=None)
team_names = df.iloc[0].dropna().tolist()
team_data = {team: [] for team in team_names}

# Step 2: Extract firstName and lastName
for _, row in df.iloc[1:].iterrows():
    for idx, cell in enumerate(row):
        if pd.isna(cell) or idx >= len(team_names):
            continue
        lines = cell.strip().split('\n')
        if not lines or len(lines[0].split()) < 2:
            continue
        first_name, last_name = lines[0].split()[:2]
        team_data[team_names[idx]].append((first_name, last_name))

# Step 3: Visualize the parsed player data
parsed_df = pd.DataFrame({
    team: pd.Series([f"{fn} {ln}" for fn, ln in players])
    for team, players in team_data.items()
})

parsed_df

,Italy,Guatemala,Brazil,Canada,Venezuela,Argentina
0,Jack D'Amore,Adrian Rodriguez,Noah Rossoni,Thomas Pretty,Nicolas Reyna,Lucas Alarcon
1,William Murray,Cormac Mulvey,Brendan Allen,Spencer Girling,Josh Whitaker,Zachary Beebe
2,Jaden Waldman,Samuel Rossoni,Gabe Duque,Perrin Root,Adriano Carpi,Megan Rapino
3,Dutch Schlegelmilch,Owen Green,Thomas Corridon,James Pretty,Emmett Nivaud,Carton Spueller
4,Spencer Derakhshan,Harry Ocampo,Luke Diedrich,Max Caro,Gustavo Reyna,Jimmy Powers
5,Arthur Khisyamov,Babayo Billy,Zach Goldfarb,Eli Zev,Zachary Delman,Massimo Marcelo
6,Ciaran Kennedy,Harrison Waller,Henry Green,Ethan Katzner,Gabe Hellmann,Alex Burtzlaff
7,Charles-Henry Nivaud,Elliot Galin,Michael Brennan,Jack Fenn,Kai Massicott,George Smith
8,Lachlan Langone,Connor Jones,Loewe Epstein,Karsten Langone,Kailan Spadea,Alan Fiore
9,Graham Tybur,Oleksandr Cherep,Reese Watkins,Chase Cammeyer,Jack Foster,Haran eiger


In [14]:
def get_email(first, last):
    first = first.strip()
    last = last.strip()

    users_ref = db.collection('users')
    candidates = users_ref.where(filter=FieldFilter("lastName", "==", last)).stream()

    matched = []
    for doc in candidates:
        user = doc.to_dict()
        print("🔍 Checking:", user.get("firstName"), "/", user.get("nickname"))
        if user.get("firstName", "").strip() == first or user.get("nickname", "").strip() == first:
            return user.get("email")

    print(f"❌ No match for first='{first}', last='{last}'")
    return None

In [15]:
team_emails = {}

# Get emails for each player and preview the result
for team_name, players in team_data.items():
    emails = []
    for first, last in players:
        email = get_email(first, last)
        if email:
            emails.append(email)
        else:
            print(f"⚠️ No email found for {first} {last} in team {team_name}")
    team_emails[team_name] = emails

# Display as a DataFrame before writing
preview_df = pd.DataFrame(dict([(team, pd.Series(emails)) for team, emails in team_emails.items()]))

preview_df

🔍 Checking: Jack / Jack
🔍 Checking: William / Will
🔍 Checking: Jaden / Jaden
🔍 Checking: Dutch / Dutch
🔍 Checking: Spencer / Spence 
🔍 Checking: Arthur / Arthur
🔍 Checking: Ciaran  / Ciarán Kennedy
🔍 Checking: Charles-Henry / Charles
🔍 Checking: Karsten / Karsten Langone
🔍 Checking: Lachlan / Lachlan
🔍 Checking: Graham / Graham
🔍 Checking: Andy / Andy
🔍 Checking: Lucas / Lucas
🔍 Checking: Sydney / Junyah - Jun - Syd 
🔍 Checking: Jack / Jack
🔍 Checking: Sebastian  / Seb
🔍 Checking: Adrian / Adrian
🔍 Checking: Cormac / Macca
🔍 Checking: Samuel / Sam
🔍 Checking: Henry / Henry
🔍 Checking: Henry / Henry
🔍 Checking: Henry / Henry
🔍 Checking: Owen / Owen
🔍 Checking: Harry / Coach Harry
🔍 Checking: Babayo / Chief
🔍 Checking: Harrison / Harrison
🔍 Checking: Elliot / Elliot Galin
🔍 Checking: Connor / Connor
🔍 Checking: Oleksandr / Sasha
🔍 Checking: Ford / Ford
🔍 Checking: Aidan / Merm
🔍 Checking: Felix / Felix
🔍 Checking: Anshuman / Anshu
🔍 Checking: Jeffrey / Jeff Weber
🔍 Checking: Sebastian  /

,Italy,Guatemala,Brazil,Canada,Venezuela,Argentina
0,damorejack46@gmail.com,adrianrod17455@gmail.com,nrossoni09@gmail.com,thomaspretty02@icloud.com,nicoreyna@gmail.com,lucas.alarcon.frias@gmail.com
1,wammurray4@gmail.com,macmulv@icloud.com,brendan.m.allen25@gmail.com,girlingspencer@gmail.com,joshjwhitaker@icloud.com,zdbeebe@gmail.com
2,jadenwaldman2028@gmail.com,srossoni07@gmail.com,gabrielduque1210@icloud.com,perrinroot@gmail.com,adrianolcarpi@gmail.com,ssalfageme@gmail.com
3,dutchschlegelmilch@gmail.com,og27wp@gmail.com,tccorridon@gmail.com,jamespretty07@gmail.com,emmettnivaud@gmail.com,muellerboys83@gmail.com
4,spencer.derakhshan5@gmail.com,harryo1o16@gmail.com,jkdiedrich@gmail.com,amaxcaro@gmail.com,greynah@gmail.com,jadenmmueller@gmail.com
5,arthur.r.k@icloud.com,babayobilly@gmail.com,mstepanis@yahoo.com,janetazev@gmail.com,zdelman2@gmail.com,maxmaurillo11@gmail.com
6,ciaran.kennedy0522@gmail.com,hrw0529@gmail.com,hg23wp@gmail.com,jmkatzner@gmail.com,gabehellmann24@gmail.com,alexburtzlaff@icloud.com
7,charlesnivaud@gmail.com,elliotgalin@gmail.com,michaeljbrennan55@gmail.com,jackfenn8@gmail.com,kai.massicott@gmail.com,gps1823@gmail.com
8,lachlantlangone@gmail.com,connormaxjones@icloud.com,loeweepstein@gmail.com,karstenlangone2@gmail.com,kailanspadea@gmail.com,alanfiore7@gmail.com
9,marietybur@gmail.com,oleksandrcherep80@gmail.com,rjwatkins2004@gmail.com,chase.cammeyer@gmail.com,jackf2461@gmail.com,eigergi2@gmail.com


In [16]:
# Step 2: Upload emails to Firestore using team document name matching

# Get the league document
league_query = db.collection('leagues').where('name', '==', 'Watermelon Cup 2026').get()
if not league_query:
    print("❌ League not found.")
else:
    league_ref = league_query[0].reference
    teams_ref = league_ref.collection('teams')

    for team_name, emails in team_emails.items():
        # Find the existing team document where name == team_name
        team_query = teams_ref.where(filter=FieldFilter("name", "==", team_name)).get()
        if not team_query:
            print(f"❌ Team '{team_name}' not found in Firestore.")
            continue

        # Update the players array
        team_doc_ref = team_query[0].reference
        team_doc_ref.update({"players": emails})
        print(f"✅ Updated team '{team_name}' with {len(emails)} players.")


c:\Users\mdj20\Documents\GitHub\watermelon-cup\.venv\Lib\site-packages\google\cloud\firestore_v1\base_collection.py:317: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


✅ Updated team 'Italy' with 14 players.
✅ Updated team 'Guatemala' with 15 players.
✅ Updated team 'Brazil' with 14 players.
✅ Updated team 'Canada' with 14 players.
✅ Updated team 'Venezuela' with 14 players.
✅ Updated team 'Argentina' with 14 players.
